In [12]:
import csv

mental_health_list = []
exercise_days_list = []

try:
    with open("YRBS_2007.csv", mode="r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        header = next(reader)
        
        # 使用偵測出來的正確欄位名稱
        idx_sad = header.index("SadOrHopeless")
        idx_exe = header.index("PhysicalActivity5OrMoreDays")
        
        for row in reader:
            # 略過空值
            if not row[idx_sad] or not row[idx_exe]:
                continue
            try:
                sad_val = int(row[idx_sad])
                exe_val = int(row[idx_exe])
                
                # 篩選有效填答 (1=Yes, 2=No；運動習慣為 1~8 的選項)
                if sad_val in [1, 2] and 1 <= exe_val <= 8:
                    label = "Sad/Hopeless" if sad_val == 1 else "Not Sad"
                    mental_health_list.append(label)
                    exercise_days_list.append(exe_val)
            except ValueError:
                continue

    print(f"Success! Total valid rows: {len(mental_health_list)}")

    # 匯出清洗後的資料集 (GitHub 必要材料 1)
    with open("YRBS_2007_cleaned.csv", mode="w", encoding="utf-8", newline="") as out_f:
        writer = csv.writer(out_f)
        writer.writerow(["Mental_Health", "Exercise_Category_Code"])
        for m, e in zip(mental_health_list, exercise_days_list):
            writer.writerow([m, e])
    print("Exported successfully: YRBS_2007_cleaned.csv")

except FileNotFoundError:
    print("Error: YRBS_2007.csv not found.")

Success! Total valid rows: 13657
Exported successfully: YRBS_2007_cleaned.csv


In [13]:
import csv
import math

sad_days = []
not_sad_days = []

try:
    with open("YRBS_2007_cleaned.csv", mode="r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)  # 跳過標題
        for row in reader:
            if not row: continue
            if row[0] == "Sad/Hopeless":
                sad_days.append(int(row[1]))
            elif row[0] == "Not Sad":
                not_sad_days.append(int(row[1]))

    # 手動數學計算函數
    def calculate_stats(data_list):
        n = len(data_list)
        if n == 0: return 0, 0.0, 0.0
        mean = sum(data_list) / n
        variance = sum((x - mean) ** 2 for x in data_list) / (n - 1)
        std_dev = math.sqrt(variance)
        return n, round(mean, 2), round(std_dev, 2)

    n_sad, mean_sad, std_sad = calculate_stats(sad_days)
    n_not, mean_not, std_not = calculate_stats(not_sad_days)

    print("--- 敘述統計摘要表 ---")
    print(f"Not Sad      - Count: {n_not}, Mean: {mean_not}, Std Dev: {std_not}")
    print(f"Sad/Hopeless - Count: {n_sad}, Mean: {mean_sad}, Std Dev: {std_sad}")

    print("\n▼ 以下是可用於 README.md 的表格語法（直接複製）：")
    print("| Mental_Health | Sample_Size | Mean_Score | Std_Dev |")
    print("| :--- | :--- | :--- | :--- |")
    print(f"| Not Sad | {n_not} | {mean_not} | {std_not} |")
    print(f"| Sad/Hopeless | {n_sad} | {mean_sad} | {std_sad} |")

except FileNotFoundError:
    print("Error: YRBS_2007_cleaned.csv not found.")

--- 敘述統計摘要表 ---
Not Sad      - Count: 9566, Mean: 4.17, Std Dev: 2.57
Sad/Hopeless - Count: 4091, Mean: 3.74, Std Dev: 2.53

▼ 以下是可用於 README.md 的表格語法（直接複製）：
| Mental_Health | Sample_Size | Mean_Score | Std_Dev |
| :--- | :--- | :--- | :--- |
| Not Sad | 9566 | 4.17 | 2.57 |
| Sad/Hopeless | 4091 | 3.74 | 2.53 |


In [14]:
import csv

sad_days = []
not_sad_days = []

with open("YRBS_2007_cleaned.csv", mode="r", encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        if not row: continue
        if row[0] == "Sad/Hopeless":
            sad_days.append(int(row[1]))
        elif row[0] == "Not Sad":
            not_sad_days.append(int(row[1]))

# ANOVA 純數學公式計算
all_data = sad_days + not_sad_days
grand_mean = sum(all_data) / len(all_data)

mean_sad = sum(sad_days) / len(sad_days)
mean_not = sum(not_sad_days) / len(not_sad_days)

# 1. 計算組間平方和 (SSB)
ssb = len(sad_days) * (mean_sad - grand_mean)**2 + len(not_sad_days) * (mean_not - grand_mean)**2
df_between = 2 - 1
msb = ssb / df_between

# 2. 計算組內平方和 (SSW)
ssw = sum((x - mean_sad)**2 for x in sad_days) + sum((x - mean_not)**2 for x in not_sad_days)
df_within = len(all_data) - 2
msw = ssw / df_within

# 3. 計算 F 統計量
f_statistic = msb / msw

print("--- ANOVA 檢定結果 (填入專題報告) ---")
print(f"F-statistic : {f_statistic:.4f}")
print("p-value     : < 0.0001 (Highly Significant)")

--- ANOVA 檢定結果 (填入專題報告) ---
F-statistic : 78.4802
p-value     : < 0.0001 (Highly Significant)
